# Multilingual NER Tokenization

This notebook handles the tokenization of the harmonized datasets and the alignment of BIO labels to sub-tokens.

## Environment Setup

Loading necessary libraries and configuring the compute device (CUDA/CPU).

In [1]:
import torch
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Tokenizer Initialization

Initializing the `xlm-roberta-base` tokenizer.
This model is chosen for its **comprehensive multilingual support** and **massive shared vocabulary**. These features are critical for the project as they enable better cross-lingual transfer between Hungarian, English, and German, even when training on a combined dataset.

In [2]:
model_id = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

## Loading Preprocessed Datasets

Loading the Interleaved, Hungarian, English, German, and  Gold_only datasets from disk.

In [3]:
interleaved_dataset = load_from_disk("../data/processed/harmonized_dataset/interleaved_dataset")
gold_only_dataset = load_from_disk("../data/processed/harmonized_dataset/gold_only_dataset")
hun_dataset = load_from_disk("../data/processed/harmonized_dataset/hun_dataset")
eng_dataset = load_from_disk("../data/processed/harmonized_dataset/eng_dataset")
ger_dataset = load_from_disk("../data/processed/harmonized_dataset/ger_dataset")

interleaved_dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 72000
    })
    validation: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 3699
    })
    test: Dataset({
        features: ['tokens', 'ner'],
        num_rows: 7374
    })
})

### Tokenization Sample

Inspecting individual examples to understand tokenization length increases.

In [4]:
example = tokenizer(interleaved_dataset['train'][0]['tokens'], is_split_into_words=True)

In [5]:
for k,v in example.items():
    print(k,"\n", v)

input_ids 
 [0, 572, 2305, 55750, 19, 61223, 98057, 295, 113419, 238, 1113, 10, 11021, 9640, 3748, 81847, 15, 87, 45176, 1388, 44200, 145, 6, 4, 295, 180, 23658, 4932, 154827, 1399, 283, 76254, 26683, 14, 60633, 1246, 32636, 14383, 10, 4620, 33670, 1330, 3522, 70016, 20, 172517, 45492, 68299, 6, 4, 295, 180, 23658, 4932, 149588, 76076, 9, 144245, 13, 295, 276, 10738, 168971, 8077, 6, 5, 2]
attention_mask 
 [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [6]:
print(f"Original splitted text:\n{interleaved_dataset['train'][0]['tokens']}", end='\n\n')
print(f"Original splitted text length:\n{len(interleaved_dataset['train'][0]['tokens'])}", end='\n\n')
print(f"Tokenized text:\n{example.tokens()}", end='\n\n')
print(f"Tokenized text length:\n{len(example.tokens())}", end='\n\n')
print(f"BIO tags:\n{interleaved_dataset['train'][0]['ner']}", end='\n\n')
print(f"BIO tags length:\n{len(interleaved_dataset['train'][0]['ner'])}")

Original splitted text:
['Hétfőn', 'folytatódik', 'az', 'Investicná', 'a', 'Rozvojová', 'Banka', '(', 'IRB', ')', 'auditja', ',', 'az', 'OTP', 'Bank', 'szakemberei', 'és', 'tanácsadói', 'ismét', 'felkeresik', 'a', 'bank', 'adatszobáját', '-', 'közölte', 'Wolf', 'László', ',', 'az', 'OTP', 'Bank', 'vezérigazgató-helyettese', 'az', 'MTI', 'érdeklődésére', '.']

Original splitted text length:
36

Tokenized text:
['<s>', '▁H', 'ét', 'fő', 'n', '▁folytat', 'ódik', '▁az', '▁Investi', 'c', 'ná', '▁a', '▁Roz', 'voj', 'ová', '▁Banka', '▁(', '▁I', 'RB', '▁)', '▁audit', 'ja', '▁', ',', '▁az', '▁O', 'TP', '▁Bank', '▁szakember', 'ei', '▁és', '▁tanács', 'adó', 'i', '▁ismét', '▁fel', 'kere', 'sik', '▁a', '▁bank', '▁adat', 'sz', 'ob', 'áját', '▁-', '▁közölte', '▁Wolf', '▁László', '▁', ',', '▁az', '▁O', 'TP', '▁Bank', '▁vezér', 'igazgató', '-', 'helyettes', 'e', '▁az', '▁M', 'TI', '▁érdeklődés', 'ére', '▁', '.', '</s>']

Tokenized text length:
67

BIO tags:
[0, 0, 0, 3, 4, 4, 4, 0, 3, 0, 0, 0, 0, 3, 4,

In [7]:
print(example.word_ids())

[None, 0, 0, 0, 0, 1, 1, 2, 3, 3, 3, 4, 5, 5, 5, 6, 7, 8, 8, 9, 10, 10, 11, 11, 12, 13, 13, 14, 15, 15, 16, 17, 17, 17, 18, 19, 19, 19, 20, 21, 22, 22, 22, 22, 23, 24, 25, 26, 27, 27, 28, 29, 29, 30, 31, 31, 31, 31, 31, 32, 33, 33, 34, 34, 35, 35, None]


## Label Alignment Function

Defining the logic to correctly map the original BIO labels to the first sub-token of each word, assigning -100 to special tokens and subsequent sub-tokens.

In [8]:
def align_labels_to_tokenized_text(ds, tokenizer, token_col, ner_col):
    tokenized_inputs = tokenizer(
        ds[token_col],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(ds[ner_col]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens like <s> or </s>
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # This is the FIRST sub-token of a new word
                label_ids.append(label[word_idx])
            else:
                # This is a follow-up sub-token of the same word
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


## Bulk Tokenization & Alignment

Applying the alignment logic to all datasets in batches.

In [9]:
tokenized_dataset = interleaved_dataset.map(
    align_labels_to_tokenized_text,
    batched=True,
    fn_kwargs={
        "tokenizer": tokenizer,
        "token_col": "tokens",
        "ner_col": "ner"
    },
    remove_columns=interleaved_dataset["train"].column_names
)


Map:   0%|          | 0/72000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3699 [00:00<?, ? examples/s]

Map:   0%|          | 0/7374 [00:00<?, ? examples/s]

In [10]:
hun_tokenized_dataset = hun_dataset.map(
    align_labels_to_tokenized_text,
    batched=True,
    fn_kwargs={
        "tokenizer": tokenizer,
        "token_col": "tokens",
        "ner_col": "ner"
    },
    remove_columns=hun_dataset["train"].column_names
)

Map:   0%|          | 0/12282 [00:00<?, ? examples/s]

Map:   0%|          | 0/940 [00:00<?, ? examples/s]

Map:   0%|          | 0/1728 [00:00<?, ? examples/s]

In [11]:
eng_tokenized_dataset = eng_dataset.map(
    align_labels_to_tokenized_text,
    batched=True,
    fn_kwargs={
        "tokenizer": tokenizer,
        "token_col": "tokens",
        "ner_col": "ner"
    },
    remove_columns=eng_dataset["train"].column_names
)

Map:   0%|          | 0/9482 [00:00<?, ? examples/s]

Map:   0%|          | 0/559 [00:00<?, ? examples/s]

Map:   0%|          | 0/546 [00:00<?, ? examples/s]

In [12]:
ger_tokenized_dataset = ger_dataset.map(
    align_labels_to_tokenized_text,
    batched=True,
    fn_kwargs={
        "tokenizer": tokenizer,
        "token_col": "tokens",
        "ner_col": "ner"
    },
    remove_columns=ger_dataset["train"].column_names
)

Map:   0%|          | 0/5100 [00:00<?, ? examples/s]

Map:   0%|          | 0/24000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

In [13]:
gold_only_tokenized_dataset = gold_only_dataset.map(
    align_labels_to_tokenized_text,
    batched=True,
    fn_kwargs={
        "tokenizer": tokenizer,
        "token_col": "tokens",
        "ner_col": "ner"
    },
    remove_columns=gold_only_dataset["train"].column_names
)

Map:   0%|          | 0/48000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3140 [00:00<?, ? examples/s]

Map:   0%|          | 0/6828 [00:00<?, ? examples/s]

## Verification

Comparing the original and aligned BIO labels to ensure counts match.

In [14]:
print(f"Original splitted text:\n{interleaved_dataset['train'][0]['tokens']}", end='\n\n')
print(f"Original splitted text length:\n{len(interleaved_dataset['train'][0]['tokens'])}", end='\n\n')
print(f"Tokenized text:\n{example.tokens()}", end='\n\n')
print(f"Tokenized text length:\n{len(example.tokens())}", end='\n\n')
print(f"BIO tags:\n{interleaved_dataset['train'][0]['ner']}", end='\n\n')
print(f"BIO tags length:\n{len(interleaved_dataset['train'][0]['ner'])}", end='\n\n')
print(f"BIO tags after tokenization:\n{tokenized_dataset['train'][0]['labels']}", end='\n\n')
print(f"BIO tags length after tokenization:\n{len(tokenized_dataset['train'][0]['labels'])}", end='\n\n')

Original splitted text:
['Hétfőn', 'folytatódik', 'az', 'Investicná', 'a', 'Rozvojová', 'Banka', '(', 'IRB', ')', 'auditja', ',', 'az', 'OTP', 'Bank', 'szakemberei', 'és', 'tanácsadói', 'ismét', 'felkeresik', 'a', 'bank', 'adatszobáját', '-', 'közölte', 'Wolf', 'László', ',', 'az', 'OTP', 'Bank', 'vezérigazgató-helyettese', 'az', 'MTI', 'érdeklődésére', '.']

Original splitted text length:
36

Tokenized text:
['<s>', '▁H', 'ét', 'fő', 'n', '▁folytat', 'ódik', '▁az', '▁Investi', 'c', 'ná', '▁a', '▁Roz', 'voj', 'ová', '▁Banka', '▁(', '▁I', 'RB', '▁)', '▁audit', 'ja', '▁', ',', '▁az', '▁O', 'TP', '▁Bank', '▁szakember', 'ei', '▁és', '▁tanács', 'adó', 'i', '▁ismét', '▁fel', 'kere', 'sik', '▁a', '▁bank', '▁adat', 'sz', 'ob', 'áját', '▁-', '▁közölte', '▁Wolf', '▁László', '▁', ',', '▁az', '▁O', 'TP', '▁Bank', '▁vezér', 'igazgató', '-', 'helyettes', 'e', '▁az', '▁M', 'TI', '▁érdeklődés', 'ére', '▁', '.', '</s>']

Tokenized text length:
67

BIO tags:
[0, 0, 0, 3, 4, 4, 4, 0, 3, 0, 0, 0, 0, 3, 4,

## Saving Tokenized Datasets

Exporting the final datasets to disk for use in model fine-tuning.

In [15]:
print(tokenized_dataset)
tokenized_dataset.save_to_disk("../data/processed/tokenized_dataset/tokenized_dataset")

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 72000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3699
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 7374
    })
})


Saving the dataset (0/1 shards):   0%|          | 0/72000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3699 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7374 [00:00<?, ? examples/s]

In [16]:
hun_tokenized_dataset.save_to_disk("../data/processed/tokenized_dataset/hun_tokenized_dataset")
eng_tokenized_dataset.save_to_disk("../data/processed/tokenized_dataset/eng_tokenized_dataset")
ger_tokenized_dataset.save_to_disk("../data/processed/tokenized_dataset/ger_tokenized_dataset")
gold_only_tokenized_dataset.save_to_disk("../data/processed/tokenized_dataset/gold_only_tokenized_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/12282 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/940 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1728 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9482 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/559 [00:00<?, ? examples/s]

...

Saving the dataset (0/1 shards):   0%|          | 0/24000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/48000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3140 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/6828 [00:00<?, ? examples/s]